In [1]:
import re
import warnings
import os
import pandas as pd
import pdfplumber

In [2]:
COLUMNS = [
    "diplome", "intitule_emploi", "missions", "categorie_emploi", "type_emploi",
    "secteur_employeur", "activite_employeur", "lieu_emploi", "revenu",
    "autre_diplome",
]
# Colonnes du tableau, dans l'ordre du pdf (10 colonnes, comme en 2024).

PARAS_TABLEAUX = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines"
}
# Même réglage que pour 2024 et 2016 : les séparations à prendre en compte sont les bordures du tableau.

TEXT_KW = {
    "x_tolerance": 1.5,
    "y_tolerance": 2
}
# Les espaces entre les mots sont très serrés (environ 2 points).
# Une faible tolérance évite de coller plusieurs mots ensemble.

# Bande verticale contenant le pied de page (« Mention … - Parcours … / Retour au sommaire / n° de page »).
# La largeur est celle de la page entière. Seule la limite haute est fixée.
FOOTER_Y_MIN = 550

**Comment ces paramètres ont été choisis.** Comme pour 2016, l'en-tête et l'encadré de stats ne sont pas lus dans une zone fixe : on lit le texte de la page entière et on s'ancre sur des mots-clés (`RÉPERTOIRE DES EMPLOIS DES DIPLÔMÉS DE MASTER`, `Mention`, `Parcours`, `Promotion`...).
La seule zone fixée est le pied de page (`FOOTER_Y_MIN`) : il est présent sur **toutes** les pages, y compris les pages « suite », et donne à la fois la mention et le parcours, ce qui permet de rattacher chaque page à sa formation et de contrôler l'en-tête.
Les particularités du pdf 2022 (formulations, pages sans en-tête, pages de notes seules) ont été repérées en inspectant le fichier (positions des mots, tableaux détectés) avec l'aide de Claude, puis validées sur l'ensemble des pages.

In [3]:
HEADER_RE = re.compile(
    r"RÉPERTOIRE DES EMPLOIS DES DIPLÔMÉS DE MASTER\s*"
    r"Mention\s+(?P<mention>.+?)\s*"
    r"Parcours\s+(?P<parcours>.+?)\s*"
    r"Promotion\s+(?P<promotion>\d{4})\s*"
    r"Situation en\s+(?P<situation>.+?)\s*-\s*Enquête à\s+(?P<delai_enquete>[^\n]+)",
    re.S,
)
# Exemple de ce qu'on cherche :
#   Mention Automatique et systèmes électriques / Parcours Gestion des réseaux d'énergie électrique /
#   Promotion 2022 / Situation en décembre 2023 - Enquête à 18 mois

# Pied de page : « Mention <mention> - Parcours <parcours> » puis « Retour au sommaire ».
FOOTER_RE = re.compile(
    r"Mention\s+(?P<mention>.+?)\s+-\s+Parcours\s+(?P<parcours>.+?)\s*Retour au sommaire",
    re.S,
)

# Un motif de recherche est associé à chaque statistique.
STATS_RE = {
    "n_concernes": r"concern\S*\s+par l['’]enquête\s*:\s*(\d+)",
    "taux_reponse": r"Taux de réponse\s*:\s*(\d+)\s*%",
    "n_en_emploi": r"En emploi\s+(\d+)",
    "n_recherche_emploi": r"En recherche d['’]emploi\s+(\d+)",
    "n_en_etudes": r"En études\s+(\d+)",
    "n_autre_situation": r"Autre situation\s+(\d+)",
    "n_total_repondants": r"Total\s+(\d+)",
    "n_emploi_stable": r"Emploi stable\s*:?\s*(\d+)",          # ":" facultatif (absent en 2022)
    "n_emploi_non_stable": r"Emploi non stable\s*:?\s*(\d+)",
}

# Pour ces catégories, une ligne absente de l'encadré signifie zéro.
# C'est vérifié sur ce pdf : stables + non stables = en emploi, et la somme des catégories = total, pour toutes les formations.
BOX_COUNTS = [
    "n_en_emploi",
    "n_recherche_emploi",
    "n_en_etudes",
    "n_autre_situation",
    "n_emploi_stable",
    "n_emploi_non_stable",
]

In [4]:
def clean(text):
    """Nettoie le texte d'une cellule et remplace certaines valeurs par None.
    Le but est de garantir que les données récupérées sont propres et homogènes avant d'être stockées."""
    if text is None:
        return None

    # Remplacer les suites d'espaces et de retours à la ligne par un espace.
    text = re.sub(r"\s+", " ", text).strip()

    # Considérer quelques valeurs comme des données manquantes :
    if text in ("", "-", "NR"):
        return None

    return text


def cle_formation(infos):
    """Clé (mention, parcours) normalisée, pour comparer l'en-tête et le pied de page
    (les apostrophes typographiques et les espaces peuvent différer)."""
    def norm(s):
        return re.sub(r"\s+", " ", s.replace("’", "'")).strip()
    return (norm(infos["mention"]), norm(infos["parcours"]))

In [5]:
def parse_header(page):
    """Renvoie les informations de l'en-tête, ou None si la page n'a pas d'en-tête de formation.
    En 2022, seule la première page d'une formation en a un : les pages « suite » n'en ont pas."""
    text = page.extract_text(**TEXT_KW) or ""  # texte de toute la page
    match = HEADER_RE.search(text)

    if not match:
        return None

    return {
        key: re.sub(r"\s+", " ", value).strip()
        for key, value in match.groupdict().items()
    }

In [6]:
def parse_footer(page):
    """Renvoie la mention et le parcours écrits en pied de page, ou None.
    Le pied de page existe sur toutes les pages : il permet d'identifier les pages « suite »."""
    text = page.crop(
        (0, FOOTER_Y_MIN, page.width, page.height)
    ).extract_text(**TEXT_KW) or ""

    match = FOOTER_RE.search(text)

    if not match:
        return None

    return {
        key: re.sub(r"\s+", " ", value).strip()
        for key, value in match.groupdict().items()
    }

In [7]:
def parse_stats(page):  # Même principe qu'en 2016 : texte de la page entière
    """Lit les statistiques présentes dans l'encadré de la première page d'une formation."""
    text = page.extract_text(**TEXT_KW) or ""

    stats = {}

    # Utilisation des expressions régulières pour chercher les stats et les stocker dans le dictionnaire :
    for key, pattern in STATS_RE.items():
        match = re.search(pattern, text)

        if match:
            stats[key] = int(match.group(1))
        else:
            stats[key] = None

    # Une ligne absente signifie zéro pour ces catégories,
    # si le total des répondants est présent.
    if stats["n_total_repondants"] is not None:
        for key in BOX_COUNTS:
            if stats[key] is None:
                stats[key] = 0

    # Idem si le PDF signale un effectif insuffisant : certaines formations n'ont pas de tableau
    # (pas assez de répondants) et c'est cette mention qui les caractérise.
    stats["effectif_insuffisant"] = "Effectif insuffisant" in text

    return stats

In [8]:
def extract_jobs(page):
    """Extrait les lignes du tableau des emplois d'une page."""
    rows = []

    # Rechercher les tableaux présents sur la page avec les paramètres spécifiés à pdfplumber
    for table in page.find_tables(PARAS_TABLEAUX):

        # Ignorer les tableaux qui n'ont pas le nombre attendu de colonnes :
        # cela permet notamment d'ignorer le petit tableau de statistiques (2 colonnes)
        if len(table.rows[0].cells) != len(COLUMNS):
            continue

        for row in table.rows:
            values = []

            # Lire et nettoyer chaque cellule de la ligne.
            for cell in row.cells:
                if cell:
                    value = clean(
                        page.crop(cell).extract_text(**TEXT_KW)
                    )
                else:
                    value = None

                values.append(value)

            # Ne pas conserver la ligne qui contient les titres des colonnes
            # (elle est répétée en haut de chaque page, y compris les pages « suite ») :
            if values[0] is not None and values[0].startswith("Diplômé"):
                continue

            # Enfin on lie les valeurs aux bonnes colonnes :
            rows.append(dict(zip(COLUMNS, values)))

    return rows

In [9]:
def extract(pdf_path, first=1, last=None):
    """Pipeline complet pour le répertoire 2022."""
    jobs = []
    formations = []
    pages_ignorees = []

    current_header = None
    current_stats = {}

    with pdfplumber.open(pdf_path) as pdf:

        # Si la dernière page n'est pas précisée, lire jusqu'à la fin du PDF.
        if last is None:
            last = len(pdf.pages)

        for page_no in range(first, last + 1):
            page = pdf.pages[page_no - 1]

            header = parse_header(page)
            footer = parse_footer(page)
            page_jobs = extract_jobs(page)

            if header is not None:
                # Page avec en-tête : nouvelle formation, on lit ses statistiques.
                stats = parse_stats(page)

                # Sécurité : l'en-tête et le pied de page doivent désigner la même formation.
                if footer is None or cle_formation(footer) != cle_formation(header):
                    warnings.warn(
                        f"Page {page_no} : en-tête et pied de page différents ({header['parcours']})."
                    )

                current_header = header
                current_stats = stats

                formations.append({
                    "page": page_no,
                    **header,
                    **stats,
                    "n_pages": 1,
                    "n_lignes_extraites": 0
                })

            else:
                # Pas d'en-tête : c'est soit une page « suite » (elle contient un tableau),
                # soit une page sans intérêt pour nous (couverture, présentation, sommaire,
                # page de notes de bas de page seules, page de fin...).
                if not page_jobs or current_header is None:
                    pages_ignorees.append(page_no)
                    continue

                # Sécurité : le pied de page doit rappeler la formation en cours.
                if footer is None or cle_formation(footer) != cle_formation(current_header):
                    warnings.warn(
                        f"Page {page_no} : le pied de page ne correspond pas à la formation en cours "
                        f"({current_header['parcours']}). Rattachement à vérifier."
                    )

                # Une page « suite » reprend l'en-tête et les statistiques de la formation en cours.
                header, stats = current_header, current_stats
                formations[-1]["n_pages"] += 1

            formations[-1]["n_lignes_extraites"] += len(page_jobs)

            for job in page_jobs:
                jobs.append({
                    "page": page_no,
                    **header,
                    **stats,
                    **job
                })

    # Garde-fou : ne pas échouer en silence si le format du pdf n'est pas celui attendu.
    if not formations:
        warnings.warn(
            "Aucune formation reconnue : le format du pdf ne correspond sans doute pas à celui attendu "
            "(vérifier HEADER_RE)."
        )
    print(f"Pages ignorées ({len(pages_ignorees)}) : {pages_ignorees}")

    # Transforme nos listes en 2 df : df est notre base globale, df_formations contient les infos propres à chaque formation
    df_formations = pd.DataFrame(formations)
    df = pd.DataFrame(jobs)

    if not df_formations.empty:
        # Contrôle : le nombre de lignes extraites doit être égal à l'effectif en emploi
        # (sauf pour les formations à « effectif insuffisant », qui n'ont pas de tableau).
        df_formations["ecart_lignes"] = (
            df_formations["n_lignes_extraites"] - df_formations["n_en_emploi"]
        )
        df_formations.loc[df_formations["effectif_insuffisant"], "ecart_lignes"] = None

    if not df.empty:
        raw = df["revenu"].fillna("")

        # Indiquer si la valeur du revenu mentionne un temps partiel.
        df["temps_partiel"] = raw.str.contains(
            "Temps partiel",
            case=False
        )

        # Convertir le revenu en nombre. Les valeurs non numériques deviennent NaN.
        df["revenu_net"] = pd.to_numeric(
            raw.str.replace(r"\s+", "", regex=True),
            errors="coerce"
        )

        # Transformer le type d'emploi en variable booléenne.
        df["emploi_stable"] = df["type_emploi"].map({
            "Stable": True,
            "Non stable": False
        })

        # Supprimer la colonne brute.
        df = df.drop(columns="revenu")

    return df, df_formations

In [10]:
from pathlib import Path

FIRST_PAGE = 1
LAST_PAGE = None

ROOT = Path.cwd().parents[1]
dossier = ROOT / "data" / "raw" / "2022"         
sortie_extract = ROOT / "data" / "interim" / "extract"
sortie_formations = ROOT / "data" / "interim" / "formations"

for fichier in os.scandir(dossier):
    if fichier.is_file() and fichier.name.lower().endswith(".pdf"):
        nom = os.path.splitext(fichier.name)[0]
        df, df_formations = extract(fichier.path, FIRST_PAGE, LAST_PAGE)
        df.to_excel(sortie_extract / f"{nom}_extract.xlsx", index=False)
        df_formations.to_excel(sortie_formations / f"{nom}_formations.xlsx", index=False)
        print(nom, df.shape, df_formations.shape)
print("fin du traitement")

Pages ignorées (26) : [1, 2, 3, 4, 5, 6, 7, 8, 21, 23, 28, 36, 39, 41, 43, 46, 62, 68, 73, 85, 92, 123, 126, 133, 135, 144]
Master2022_ALL-SHS_P (838, 28) (87, 19)
Pages ignorées (37) : [1, 2, 3, 4, 5, 6, 7, 8, 17, 41, 43, 46, 49, 55, 58, 62, 66, 68, 73, 75, 93, 97, 101, 133, 138, 142, 144, 150, 152, 158, 160, 165, 167, 184, 190, 193, 194]
Master2022_DEG_P (1382, 28) (102, 19)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20460\212907862.py:29: UserWarning: Page 7 : en-tête et pied de page différents (Cellular, integrative translational neurosciences).
  warnings.warn(


Pages ignorées (11) : [1, 2, 3, 4, 5, 6, 12, 16, 38, 56, 57]
Master2022_Sante_P (388, 28) (29, 19)
Pages ignorées (23) : [1, 2, 3, 4, 5, 6, 7, 8, 29, 33, 39, 53, 55, 58, 61, 64, 70, 78, 85, 94, 97, 105, 106]
Master2022_ST_P (503, 28) (63, 19)
fin du traitement
